# Feature Engineering
This notebook focuses on the **Feature Engineering**, the goal is to make the dataset ready for models, final feature engineering will be under src/feature_engineering.py
The data is already cleaned and loaded

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning import clean_data

df = clean_data()
df.shape, df.info()


C:\Users\taula\OneDrive\Desktop\Sem7\DSPRO\hospital-readmission-predictor\src\data_cleaning.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_data['max_glu_serum'].fillna(0, inplace=True)
C:\Users\taula\OneDrive\Desktop\Sem7\DSPRO\hospital-readmission-predictor\src\data_cleaning.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object o

<class 'pandas.core.frame.DataFrame'>
Index: 101742 entries, 0 to 101765
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   race                   101742 non-null  object
 1   gender                 101742 non-null  object
 2   age                    101742 non-null  object
 3   admission_type         101742 non-null  object
 4   discharge_disposition  101742 non-null  object
 5   admission_source       101742 non-null  object
 6   time_in_hospital       101742 non-null  int64 
 7   medical_specialty      101742 non-null  object
 8   num_lab_procedures     101742 non-null  int64 
 9   num_procedures         101742 non-null  int64 
 10  num_medications        101742 non-null  int64 
 11  number_outpatient      101742 non-null  int64 
 12  number_emergency       101742 non-null  int64 
 13  number_inpatient       101742 non-null  int64 
 14  diag_1                 101742 non-null  object
 15  diag_

((101742, 34), None)

## ICD-9 Diagnosis Categorization

In [2]:
def categorize_icd9(code):
    """Map ICD-9 diagnosis codes to major medical categories."""
    if pd.isna(code):
        return 'Unknown'
    code_str = str(code)

    # Handle E and V codes first
    if code_str.startswith('E'):
        return 'External causes of injury'
    elif code_str.startswith('V'):
        return 'Supplemental classification'

    # Try numeric conversion for standard ICD-9 codes
    try:
        code_val = float(code_str.split('.')[0])
    except ValueError:
        return 'Unknown'

    # Map ranges to categories
    if 1 <= code_val <= 139:
        return 'Infectious and parasitic diseases'
    elif 140 <= code_val <= 239:
        return 'Neoplasms'
    elif 240 <= code_val <= 279:
        return 'Endocrine, nutritional and metabolic diseases, and immunity disorders'
    elif 280 <= code_val <= 289:
        return 'Diseases of the blood and blood-forming organs'
    elif 290 <= code_val <= 319:
        return 'Mental disorders'
    elif 320 <= code_val <= 389:
        return 'Diseases of the nervous system and sense organs'
    elif 390 <= code_val <= 459:
        return 'Diseases of the circulatory system'
    elif 460 <= code_val <= 519:
        return 'Diseases of the respiratory system'
    elif 520 <= code_val <= 579:
        return 'Diseases of the digestive system'
    elif 580 <= code_val <= 629:
        return 'Diseases of the genitourinary system'
    elif 630 <= code_val <= 679:
        return 'Complications of pregnancy, childbirth, and the puerperium'
    elif 680 <= code_val <= 709:
        return 'Diseases of the skin and subcutaneous tissue'
    elif 710 <= code_val <= 739:
        return 'Diseases of the musculoskeletal system and connective tissue'
    elif 740 <= code_val <= 759:
        return 'Congenital anomalies'
    elif 760 <= code_val <= 779:
        return 'Certain conditions originating in the perinatal period'
    elif 780 <= code_val <= 799:
        return 'Symptoms, signs, and ill-defined conditions'
    elif 800 <= code_val <= 999:
        return 'Injury and poisoning'
    else:
        return 'Unknown'

# Apply to all 3 diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[f'{col}_category'] = df[col].apply(categorize_icd9)

print("Diagnosis codes categorized into medical groups.")
df[[ 'diag_1', 'diag_1_category', 'diag_2_category', 'diag_3_category']].head()


Diagnosis codes categorized into medical groups.


,diag_1,diag_1_category,diag_2_category,diag_3_category
0,250.83,"Endocrine, nutritional and metabolic diseases,...",Unknown,Unknown
1,276,"Endocrine, nutritional and metabolic diseases,...","Endocrine, nutritional and metabolic diseases,...","Endocrine, nutritional and metabolic diseases,..."
2,648,"Complications of pregnancy, childbirth, and th...","Endocrine, nutritional and metabolic diseases,...",Supplemental classification
3,8,Infectious and parasitic diseases,"Endocrine, nutritional and metabolic diseases,...",Diseases of the circulatory system
4,197,Neoplasms,Neoplasms,"Endocrine, nutritional and metabolic diseases,..."


In [3]:
df.groupby('diag_1_category')['readmitted'].mean().mul(100).sort_values(ascending=False)

diag_1_category
Diseases of the blood and blood-forming organs                           53.218495
Supplemental classification                                              50.304136
Diseases of the respiratory system                                       50.216201
Endocrine, nutritional and metabolic diseases, and immunity disorders    49.629112
Diseases of the skin and subcutaneous tissue                             48.102767
Diseases of the circulatory system                                       47.107302
Mental disorders                                                         46.595933
Diseases of the digestive system                                         45.927454
Infectious and parasitic diseases                                        45.556358
Symptoms, signs, and ill-defined conditions                              44.617601
Injury and poisoning                                                     44.334481
Diseases of the genitourinary system                                   

In [4]:
# Drop raw diagnosis columns (we now use categorized versions)
df = df.drop(columns=['diag_1', 'diag_2', 'diag_3'])
print("Dropped raw diag_1–3 columns; using categorized versions instead.")

Dropped raw diag_1–3 columns; using categorized versions instead.


In [5]:
df

,race,gender,age,admission_type,discharge_disposition,admission_source,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,pioglitazone,rosiglitazone,acarbose,insulin,change,diabetesMed,readmitted,diag_1_category,diag_2_category,diag_3_category
0,Caucasian,Female,[0-10),Not Available,Not Mapped,Physician Referral,1,Pediatrics-Endocrinology,41,0,...,No,No,No,No,False,No,False,"Endocrine, nutritional and metabolic diseases,...",Unknown,Unknown
1,Caucasian,Female,[10-20),Emergency,Discharged to home,Emergency Room,3,?,59,0,...,No,No,No,Up,True,Yes,True,"Endocrine, nutritional and metabolic diseases,...","Endocrine, nutritional and metabolic diseases,...","Endocrine, nutritional and metabolic diseases,..."
2,AfricanAmerican,Female,[20-30),Emergency,Discharged to home,Emergency Room,2,?,11,5,...,No,No,No,No,False,Yes,False,"Complications of pregnancy, childbirth, and th...","Endocrine, nutritional and metabolic diseases,...",Supplemental classification
3,Caucasian,Male,[30-40),Emergency,Discharged to home,Emergency Room,2,?,44,1,...,No,No,No,Up,True,Yes,False,Infectious and parasitic diseases,"Endocrine, nutritional and metabolic diseases,...",Diseases of the circulatory system
4,Caucasian,Male,[40-50),Emergency,Discharged to home,Emergency Room,1,?,51,0,...,No,No,No,Steady,True,Yes,False,Neoplasms,Neoplasms,"Endocrine, nutritional and metabolic diseases,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,AfricanAmerican,Male,[70-80),Emergency,Discharged/transferred to SNF,Emergency Room,3,?,51,0,...,No,No,No,Down,True,Yes,True,"Endocrine, nutritional and metabolic diseases,...",Mental disorders,Diseases of the circulatory system
101762,AfricanAmerican,Female,[80-90),Emergency,Discharged/transferred to ICF,Transfer from a Skilled Nursing Facility (SNF),5,?,33,3,...,No,No,No,Steady,False,Yes,False,Diseases of the digestive system,"Endocrine, nutritional and metabolic diseases,...","Symptoms, signs, and ill-defined conditions"
101763,Caucasian,Male,[70-80),Emergency,Discharged to home,Emergency Room,1,?,53,0,...,No,No,No,Down,True,Yes,False,Infectious and parasitic diseases,Diseases of the genitourinary system,Mental disorders
101764,Caucasian,Female,[80-90),Urgent,Discharged/transferred to SNF,Emergency Room,10,Surgery-General,45,2,...,Steady,No,No,Up,True,Yes,False,Injury and poisoning,Diseases of the blood and blood-forming organs,Injury and poisoning


## Encode Features

### Binary

In [6]:
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
df['change'] = df['change'].astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)
df['readmitted'] = df['readmitted'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101742 entries, 0 to 101765
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   race                   101742 non-null  object
 1   gender                 101742 non-null  int64 
 2   age                    101742 non-null  object
 3   admission_type         101742 non-null  object
 4   discharge_disposition  101742 non-null  object
 5   admission_source       101742 non-null  object
 6   time_in_hospital       101742 non-null  int64 
 7   medical_specialty      101742 non-null  object
 8   num_lab_procedures     101742 non-null  int64 
 9   num_procedures         101742 non-null  int64 
 10  num_medications        101742 non-null  int64 
 11  number_outpatient      101742 non-null  int64 
 12  number_emergency       101742 non-null  int64 
 13  number_inpatient       101742 non-null  int64 
 14  number_diagnoses       101742 non-null  int64 
 15  max_g

### Ordered Categories

In [7]:
# Check unique values for ordinal (ordered) features
print("Unique values in A1Cresult:")
print(df['A1Cresult'].unique(), "\n")

print("Unique values in max_glu_serum:")
print(df['max_glu_serum'].unique())


Unique values in A1Cresult:
[0 '>7' '>8' 'Norm'] 

Unique values in max_glu_serum:
[0 '>300' 'Norm' '>200']


In [8]:
map_a1c = {
    0: 0,
    'None': 0,
    'Norm': 1,
    '>7': 2,
    '>8': 3
}

map_glu = {
    0: 0,
    'None': 0,
    'Norm': 1,
    '>200': 2,
    '>300': 3
}

df['A1Cresult'] = df['A1Cresult'].map(map_a1c)
df['max_glu_serum'] = df['max_glu_serum'].map(map_glu)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101742 entries, 0 to 101765
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   race                   101742 non-null  object
 1   gender                 101742 non-null  int64 
 2   age                    101742 non-null  object
 3   admission_type         101742 non-null  object
 4   discharge_disposition  101742 non-null  object
 5   admission_source       101742 non-null  object
 6   time_in_hospital       101742 non-null  int64 
 7   medical_specialty      101742 non-null  object
 8   num_lab_procedures     101742 non-null  int64 
 9   num_procedures         101742 non-null  int64 
 10  num_medications        101742 non-null  int64 
 11  number_outpatient      101742 non-null  int64 
 12  number_emergency       101742 non-null  int64 
 13  number_inpatient       101742 non-null  int64 
 14  number_diagnoses       101742 non-null  int64 
 15  max_g

In [9]:
#One-hot encoding for remaining categorical features
categorical_cols = [
    'race', 'age', 'admission_type', 'discharge_disposition',
    'admission_source', 'diag_1_category', 'diag_2_category',
    'diag_3_category', 'medical_specialty'
]
 # Important type int instead of bool
 # Drop first column for dummy trap
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

print("Shape after encoding:", df.shape)
df.info()

Shape after encoding: (101742, 211)
<class 'pandas.core.frame.DataFrame'>
Index: 101742 entries, 0 to 101765
Columns: 211 entries, gender to medical_specialty_Urology
dtypes: int64(200), object(11)
memory usage: 164.6+ MB


In [10]:
# Find all columns that are still non-numeric
non_numeric = df.select_dtypes(include=['object']).columns
print("Columns still non-numeric:")
print(non_numeric.tolist())


Columns still non-numeric:
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'insulin']


In [11]:
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'insulin'
]

med_map = {
    'No': 0,        # not prescribed
    'Down': 1,      # dosage decreased
    'Steady': 2,    # dosage unchanged
    'Up': 3         # dosage increased
}

# Apply mapping
for col in med_cols:
    df[col] = df[col].map(med_map)

print("Encoded medication-related columns.")
print(df[med_cols].head())


Encoded medication-related columns.
   metformin  repaglinide  nateglinide  chlorpropamide  glimepiride  \
0          0            0            0               0            0   
1          0            0            0               0            0   
2          0            0            0               0            0   
3          0            0            0               0            0   
4          0            0            0               0            0   

   glipizide  glyburide  pioglitazone  rosiglitazone  acarbose  insulin  
0          0          0             0              0         0        0  
1          0          0             0              0         0        3  
2          2          0             0              0         0        0  
3          0          0             0              0         0        3  
4          2          0             0              0         0        2  


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101742 entries, 0 to 101765
Columns: 211 entries, gender to medical_specialty_Urology
dtypes: int64(211)
memory usage: 164.6 MB


In [13]:
#Explore core numeric columns before feature engineering
num_cols = [
    'num_medications', 'number_diagnoses',
    'number_inpatient', 'number_emergency',
    'number_outpatient', 'time_in_hospital'
]

print(df[num_cols].describe().round(2))

# unique counts and max
for col in num_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", sorted(df[col].unique())[:15], "...")
    print("Max:", df[col].max())


       num_medications  number_diagnoses  number_inpatient  number_emergency  \
count        101742.00         101742.00         101742.00         101742.00   
mean             16.02              7.42              0.64              0.20   
std               8.13              1.93              1.26              0.93   
min               1.00              1.00              0.00              0.00   
25%              10.00              6.00              0.00              0.00   
50%              15.00              8.00              0.00              0.00   
75%              20.00              9.00              1.00              0.00   
max              81.00             16.00             21.00             76.00   

       number_outpatient  time_in_hospital  
count          101742.00         101742.00  
mean                0.37              4.40  
std                 1.27              2.99  
min                 0.00              1.00  
25%                 0.00              2.00  
50%      

In [14]:
# Derived clinical & behavioral features

# 0=low(≤5), 1=moderate(6-10), 2=high(11-20), 3=very high(>20)
df['polypharmacy_level'] = pd.cut(
    df['num_medications'],
    bins=[0, 5, 10, 20, np.inf],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype(int)

# 0=low(≤3), 1=moderate(4-6), 2=high(7-9), 3=severe(≥10)
df['comorbidity_score'] = pd.cut(
    df['number_diagnoses'],
    bins=[0, 3, 6, 9, np.inf],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype(int)

# Total hospital contacts
df['total_visits'] = (
    df['number_inpatient'] + df['number_emergency'] + df['number_outpatient']
)

# Binary flags for hospital use
df['had_inpatient']   = (df['number_inpatient']   > 0).astype(int)
df['had_emergency']   = (df['number_emergency']   > 0).astype(int)
df['had_outpatient']  = (df['number_outpatient']  > 0).astype(int)

# Frequent visitor flag (≥5 contacts)
df['frequent_visitor'] = (df['total_visits'] >= 5).astype(int)

# 0=short(1-3 days), 1=medium(4-6 days), 2=long(7-10 days), 3=very long(>10 days)
df['stay_length_cat'] = pd.cut(
    df['time_in_hospital'],
    bins=[0, 3, 6, 10, np.inf],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype(int)

# Drop original columns
cols_to_drop = [
    'num_medications', 'number_diagnoses',
    'number_inpatient', 'number_emergency',
    'number_outpatient', 'time_in_hospital'
]
df = df.drop(columns=cols_to_drop)

print("Current shape:", df.shape)
df.head()


Current shape: (101742, 213)


,gender,num_lab_procedures,num_procedures,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,...,medical_specialty_SurgicalSpecialty,medical_specialty_Urology,polypharmacy_level,comorbidity_score,total_visits,had_inpatient,had_emergency,had_outpatient,frequent_visitor,stay_length_cat
0,1,41,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,59,0,0,0,0,0,0,0,0,...,0,0,2,2,0,0,0,0,0,0
2,1,11,5,0,0,0,0,0,0,0,...,0,0,2,1,3,1,0,1,0,0
3,0,44,1,0,0,0,0,0,0,0,...,0,0,2,2,0,0,0,0,0,0
4,0,51,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0


## Datasplit before using Scaler

In [15]:
from sklearn.model_selection import train_test_split

# x are features, y is target
X = df.drop('readmitted', axis=1)
y = df['readmitted']

# split 70% train, 30% val and test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# split the 20% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)



## Normalization

In [16]:
from sklearn.preprocessing import StandardScaler

# continuous variables to scale
to_scale = ['num_lab_procedures', 'num_procedures', 'total_visits']

# Initialize scaler
scaler = StandardScaler()

# Fit and transform
X_train[to_scale] = scaler.fit_transform(X_train[to_scale])
X_val[to_scale] = scaler.transform(X_val[to_scale])
X_test[to_scale] = scaler.transform(X_test[to_scale])

df[to_scale].describe().round(2)


,num_lab_procedures,num_procedures,total_visits
count,101742.00,101742.00,101742.00
mean,43.10,1.34,1.20
std,19.67,1.71,2.29
min,1.00,0.00,0.00
25%,31.00,0.00,0.00
50%,44.00,1.00,0.00
75%,57.00,2.00,2.00
max,132.00,6.00,80.00


## Feature Engineering Results

- All categorical, ordinal, and medication-related columns have been encoded into numeric form.
- Clinically meaningful features were derived:
  - polypharmacy_level
  - comorbidity_score
  - total_visits, num_lab_procedures, num_procedures scaled
  - frequent_visitor
  - stay_length_cat
- The dataset is now **fully numeric, standardized, and ready for modeling**.
